In [1]:
import pandas as pd
import numpy as np

train_df = pd.read_csv("../data/processed/model_train.csv")
val_df = pd.read_csv("../data/processed/model_val.csv")
test_df = pd.read_csv("../data/processed/model_test.csv")

c:\Users\loaye\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
c:\Users\loaye\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


In [11]:
train_df

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V27,V28,Amount,Class,Hour,Day,Hour_sin,Hour_cos,Time_of_day,amount_log
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,0.133558,-0.021053,149.62,0,0,0,0.000000e+00,1.0,Night,5.014760
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.008983,0.014724,2.69,0,0,0,0.000000e+00,1.0,Night,1.305626
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,-0.055353,-0.059752,378.66,0,0,0,0.000000e+00,1.0,Night,5.939276
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,0.062723,0.061458,123.50,0,0,0,0.000000e+00,1.0,Night,4.824306
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,0.219422,0.215153,69.99,0,0,0,0.000000e+00,1.0,Night,4.262539
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
198603,132904.0,-2.348563,-2.118430,0.968801,0.171939,1.476173,-0.066357,1.316844,-0.029965,0.564404,...,-0.234552,-0.040684,406.29,0,12,1,1.224647e-16,-1.0,Afternoon,6.009525
198604,132905.0,-0.473957,1.460497,-0.687928,-0.454381,0.375335,-1.071949,0.634231,0.306173,-0.096931,...,0.211713,0.084178,8.99,0,12,1,1.224647e-16,-1.0,Afternoon,2.301585
198605,132905.0,0.183706,0.752492,0.120348,-0.731923,0.533554,-1.237835,1.021313,-0.329895,-0.316811,...,-0.013347,0.024715,1.00,0,12,1,1.224647e-16,-1.0,Afternoon,0.693147
198606,132905.0,1.774140,-1.081432,-0.092484,0.194936,-0.816991,0.975553,-1.165119,0.434134,1.988660,...,0.030006,-0.027821,87.00,0,12,1,1.224647e-16,-1.0,Afternoon,4.477337


In [12]:
numeric_features = train_df.drop(columns=['Class']).select_dtypes(include=[np.number]).columns.tolist()
categorical_features = train_df.drop(columns=['Class']).select_dtypes(include=[object]).columns.tolist()

C:\Users\loaye\AppData\Local\Temp\ipykernel_16296\2758812355.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = train_df.drop(columns=['Class']).select_dtypes(include=[object]).columns.tolist()


In [15]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

numeric_pipeline = Pipeline([
    ('scaler', StandardScaler())
])

categorical_pipeline = Pipeline([
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_pipeline, numeric_features),
        ('cat', categorical_pipeline, categorical_features)
    ]
)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

pipeline_logreg = Pipeline([
    ('preprocessor', preprocessor),
    ('logistic_regression', LogisticRegression(class_weight='balanced', max_iter=1000))
])

X_train = train_df.drop(columns=['Class'])
y_train = train_df['Class']
pipeline_logreg.fit(X_train, y_train)
X_test = test_df.drop(columns=['Class'])
y_test = test_df['Class']
logreg_pred = pipeline_logreg.predict(X_test)
print(classification_report(y_test, logreg_pred))
print(confusion_matrix(y_test, logreg_pred))
print(roc_auc_score(y_test, logreg_pred))

              precision    recall  f1-score   support

           0       1.00      0.96      0.98     42507
           1       0.03      0.88      0.05        52

    accuracy                           0.96     42559
   macro avg       0.51      0.92      0.52     42559
weighted avg       1.00      0.96      0.98     42559

[[40896  1611]
 [    6    46]]
0.9233578722780501


In [ ]:
logreg_probabilities = pipeline_logreg.predict_proba(X_test)[:,1]
logreg_probabilities

array([0.02134331, 0.0039195 , 0.10638063, ..., 0.02087004, 0.00959157,
       0.02261682])

In [25]:
from sklearn.ensemble import RandomForestClassifier

pipeline_rf = Pipeline([
    ('preprocessor', preprocessor),
    ('random_forest', RandomForestClassifier(n_estimators=300, class_weight='balanced', max_depth=8, random_state=42))
])
pipeline_rf.fit(X_train, y_train)
rf_pred = pipeline_rf.predict(X_test)
print(classification_report(y_test, rf_pred))
print(confusion_matrix(y_test, rf_pred))
print(roc_auc_score(y_test, rf_pred))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00     42507
           1       0.85      0.75      0.80        52

    accuracy                           1.00     42559
   macro avg       0.92      0.87      0.90     42559
weighted avg       1.00      1.00      1.00     42559

[[42500     7]
 [   13    39]]
0.8749176606206037


In [26]:
import xgboost as xgb

pipeline_xgb = Pipeline([
    ('preprocessor', preprocessor),
    ('xgboost', xgb.XGBClassifier(n_estimators=300, max_depth=8, scale_pos_weight=4, random_state=42))
])
pipeline_xgb.fit(X_train, y_train)
xgb_pred = pipeline_xgb.predict(X_test)
print(classification_report(y_test, xgb_pred))
print(confusion_matrix(y_test, xgb_pred))
print(roc_auc_score(y_test, xgb_pred))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00     42507
           1       0.90      0.73      0.81        52

    accuracy                           1.00     42559
   macro avg       0.95      0.87      0.90     42559
weighted avg       1.00      1.00      1.00     42559

[[42503     4]
 [   14    38]]
0.8653375643106747


In [27]:
from sklearn.metrics import recall_score, precision_score, f1_score

comparison_df = pd.DataFrame({
    'Model': ['Logistic Regression', 'Random Forest', 'XGBoost'],
    'Recall': [recall_score(y_test, logreg_pred), recall_score(y_test, rf_pred), recall_score(y_test, xgb_pred)],
    'Precision': [precision_score(y_test, logreg_pred), precision_score(y_test, rf_pred), precision_score(y_test, xgb_pred)],
    'F1 Score': [f1_score(y_test, logreg_pred), f1_score(y_test, rf_pred), f1_score(y_test, xgb_pred)],
    'ROC AUC': [roc_auc_score(y_test, logreg_pred), roc_auc_score(y_test, rf_pred), roc_auc_score(y_test, xgb_pred)]
})
comparison_df

,Model,Recall,Precision,F1 Score,ROC AUC
0,Logistic Regression,0.884615,0.027761,0.053833,0.923358
1,Random Forest,0.750000,0.847826,0.795918,0.874918
2,XGBoost,0.730769,0.904762,0.808511,0.865338


with no hyperparameter tuning, xgboost model gets the best precision but worst recall, while logistic regression gets best recall but a very bad precision score (flags most transactions as fraud when they're not). 